Ejecuta la siguiente celda e introduce la contraseña facilitada en el curso para desbloquear el cuaderno.


In [ ]:
%pip install -q git+https://github.com/dcrespons14/cam-estrategia-aula.git
from cam_estrategia import desbloquear_cuaderno
desbloquear_cuaderno()


# Modelos de degradación

En este cuaderno construiremos **modelos empíricos de degradación** a partir de los Grandes Premios de México de 2023, 2024 y 2025.

Ajustaremos tres modelos:
- Lineal.
- Polinómico.
- Por fases.

Los datos vienen de [OpenF1](https://openf1.org), una API pública y gratuita.

## Inicialización

### Importar librerías

Importamos las librerías relevantes.

In [ ]:
# Librerías propias (ver src/race_analysis y src/race_models)
from race_analysis.ingestion.openf1 import load_race, get
from race_analysis.ingestion.laps import clean_data
from race_analysis.tyre_deg.degradation import usable_laps, compare_models
from race_analysis.tyre_deg.degradation_plots import plot_fits, plot_track_evolution
from race_analysis.tyre_deg.pace import fit_pooled_race_pace
from race_models.f1.models import base_pace_from_race_pace
from race_models.f1.analysis.race_plots import plot_base_pace, plot_tyre_model

# Librerías de Python
import pandas as pd

### Descargar los datos

`load_race` descarga las vueltas, stints, compuestos y otros datos relevantes de la API.

In [ ]:
TRACK = "Mexico"
YEARS = [2023, 2024, 2025]

races = {year: load_race(year, TRACK) for year in YEARS}

for year, race in races.items():
    print(f"{year}: {race.total_laps} vueltas, {len(race.stints)} stints.")

### Limpiar los datos

Limpiamos los datos descartando la primera vuelta de la carrera, vueltas bajo neutralización y vueltas lentas, entre otros.

In [ ]:
laps = pd.concat([clean_data(race) for race in races.values()], ignore_index=True)

### Efecto del combustible

Corregimos el efecto del combustible para que la pérdida de peso no esconda parte de la degradación.

In [ ]:
FUEL_EFFECT = 0.05  # Valor aproximado, estimación habitual en F1

# A cada vuelta le restamos FUEL_EFFECT por cada vuelta de combustible que todavía queda en el coche
laps["fuel_corrected"] = laps["lap_time"] - FUEL_EFFECT * laps["laps_remaining"]

### Evolución de la pista

También calculamos la evolución de la pista.

In [ ]:
paces = fit_pooled_race_pace(laps)

plot_track_evolution(laps, paces, title="México 2023-2025 · Evolución de la pista");

## Modelos

Los tres modelos explican cada vuelta de la misma forma:

$$\text{tiempo por vuelta} = \text{ritmo base del coche} + \text{ritmo del neumático (compuesto, vida)} + \text{evolución de la pista} \times \text{vuelta}$$

- Los tiempos ya están corregidos con `FUEL_EFFECT`.
- Cada año tiene su propio **ritmo base** por coche y su propia **evolución de la pista** (`TRACK_EVO`), los segundos que cambia cada vuelta a lo largo de la carrera.
- Los neumáticos son **los mismos los tres años**.

In [ ]:
tlas = {driver["driver_number"]: driver["name_acronym"] for driver in get("drivers", session_key=races[2025].session_key)}


### Lineal

Cada vuelta de uso, el neumático pierde siempre lo mismo: $f(\text{compuesto, vida}) = \text{diferencia del compuesto} + \text{pendiente} \times (\text{vida} - 1)$.

In [ ]:
linear = fit_pooled_race_pace(laps, model="lineal")

In [ ]:
plot_tyre_model(linear[2025].tyre_curves(), max_life=45, title="México 2023-2025 · Modelo lineal");

In [ ]:
plot_base_pace(base_pace_from_race_pace(linear[2025], tlas), title="México 2025 · Ritmo base");

### Polinómico

La degradación puede acelerarse o frenarse con la edad del neumático: $f(\text{compuesto, vida}) = \text{diferencia del compuesto} + c_1 t + c_2 t^2$, con $t = \text{vida} - 1$.

In [ ]:
polynomial = fit_pooled_race_pace(laps, model="polinómico", degree=2)

plot_tyre_model(polynomial[2025].tyre_curves(), max_life=45, title="México 2023-2025 · Modelo polinómico");

### Por fases

Calentamiento, fase estable y caída. El modelo busca los puntos de cambio de cada compuesto y solo añade una fase si los datos la justifican.

In [ ]:
phases = fit_pooled_race_pace(laps, model="por fases")

plot_tyre_model(phases[2025].tyre_curves(), max_life=45, title="México 2023-2025 · Modelo por fases");

## Los tres modelos

En el siguiente gráfico comparamos los tres modelos con el neumático blando.

In [ ]:
soft = usable_laps(laps)
soft = soft[soft["compound"] == "SOFT"]
table, fits = compare_models(soft)

plot_fits(soft, fits, title="México 2025 · Blando");